# SurfaceTarget-CRC — phase 01 — patch 1 2

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.2
# TCGA Dataset Discovery and Metadata Retrieval
# ============================================================

from pathlib import Path
import requests
import json
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

GDC_API = "https://api.gdc.cancer.gov"

PROJECTS = [
    "TCGA-COAD",
    "TCGA-READ"
]

# ------------------------------------------------------------
# 2. Retrieve project metadata
# ------------------------------------------------------------

project_inventory = {
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": "NCI Genomic Data Commons",
    "api": GDC_API,
    "projects": {}
}

print("=" * 65)
print("SURFACETARGET-CRC")
print("TCGA DATASET DISCOVERY")
print("=" * 65)

for project_id in PROJECTS:

    print(f"\nInvestigating: {project_id}")
    print("-" * 50)

    endpoint = f"{GDC_API}/projects/{project_id}"

    try:

        response = requests.get(
            endpoint,
            params={"expand": "summary,summary.data_categories"},
            timeout=30
        )

        response.raise_for_status()

        data = response.json()["data"]

        summary = data.get("summary", {})

        case_count = summary.get("case_count", "Unknown")

        categories = summary.get("data_categories", [])

        project_inventory["projects"][project_id] = {
            "name": data.get("name"),
            "primary_site": data.get("primary_site"),
            "case_count_all": case_count,
            "data_categories": categories,
            "status": "metadata_retrieved"
        }

        print(f"Project: {data.get('name')}")
        print(f"Primary Site: {data.get('primary_site')}")
        print(f"Total Registered Cases: {case_count}")

        print("\nAvailable Data Categories:")

        if categories:

            for category in categories:

                name = category.get("data_category", "Unknown")
                count = category.get("case_count", "Unknown")

                print(f"  - {name}: {count} cases")

        else:

            print("  No category summary returned.")

        print("\nStatus: SUCCESS")

    except Exception as e:

        project_inventory["projects"][project_id] = {
            "status": "error",
            "error": str(e)
        }

        print(f"Status: ERROR — {e}")

# ------------------------------------------------------------
# 3. Save metadata
# ------------------------------------------------------------

output_path = METADATA_DIR / "tcga_project_inventory.json"

with open(output_path, "w") as f:

    json.dump(
        project_inventory,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DATASET DISCOVERY SUMMARY")
print("=" * 65)

for project_id, info in project_inventory["projects"].items():

    print(
        f"{project_id}: "
        f"{info.get('case_count_all', 'N/A')} cases "
        f"({info['status']})"
    )

print("\nMetadata saved to:")
print(output_path)

print("\nNOTE:")
print("These counts include all registered project cases.")
print("They are NOT MSS-specific cohort counts.")
print("No molecular datasets have been downloaded.")

print("\nPATCH 1.2 COMPLETE")